In [8]:
import subprocess

for pkg in ["folium", "ipywidgets"]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        subprocess.check_call(["uv", "add", pkg])

import folium, ipywidgets
print("folium", folium.__version__, " | ipywidgets", ipywidgets.__version__)

folium 0.20.0  | ipywidgets 8.1.8


In [11]:
import json
import geopandas as gpd
import pandas as pd
from shapely.geometry import shape, mapping

DATASET_CSV  = "./export-dataset.csv"
ZONES_FILE   = "./zones_filtered.geojson"
MAPPING_CSV  = "./geozones-mapping-output.csv"

# ── Load matching output ──────────────────────────────────────────────────────
mapping_df = pd.read_csv(MAPPING_CSV)
print(f"Mapping: {len(mapping_df)} rows")

# ── Load dataset bboxes (only the columns we need) ───────────────────────────
df = pd.read_csv(DATASET_CSV, sep=";", dtype=str, usecols=["id", "title", "spatial.geom"])
df["geometry"] = df["spatial.geom"].apply(
    lambda raw: shape(json.loads(raw)) if isinstance(raw, str) and raw.strip() else None
)
datasets = df[df["geometry"].notna()][["id", "title", "geometry"]].copy()
print(f"Datasets with bbox: {len(datasets)}")

# ── Load only the zones we actually matched (no OBSOLETE_REGIONS needed) ─────
needed_zone_ids = set(mapping_df["zone_id"])
zones_raw = gpd.read_file(ZONES_FILE, engine="pyogrio")[["id", "geometry"]]
zones_raw["id"] = zones_raw["id"].str.replace(r"@.*$", "", regex=True)
zones = zones_raw[zones_raw["id"].isin(needed_zone_ids)].set_index("id")
print(f"Zones loaded: {len(zones)} unique zones")

Mapping: 32345 rows


Datasets with bbox: 45431


Zones loaded: 2248 unique zones


In [12]:
import folium
from ipywidgets import Text, Button, Output, VBox, HBox, Label
from shapely.geometry import mapping
import IPython.display as ipd

datasets_idx = datasets.set_index("id")
mapping_idx  = mapping_df.set_index("id")

out = Output()

def geojson_layer(geom, color, label):
    return folium.GeoJson(
        {"type": "Feature", "geometry": mapping(geom)},
        name=label,
        style_function=lambda _: {
            "color": color, "fillColor": color,
            "fillOpacity": 0.15, "weight": 2,
        },
    )

def show_dataset(dataset_id):
    out.clear_output()
    with out:
        dataset_id = dataset_id.strip()
        if not dataset_id:
            return
        if dataset_id not in datasets_idx.index:
            print(f"ID not found: {dataset_id!r}")
            return

        row = datasets_idx.loc[dataset_id]
        bbox_geom = row["geometry"]
        b = bbox_geom.bounds
        center = [(b[1]+b[3])/2, (b[0]+b[2])/2]

        m = folium.Map(location=center, zoom_start=8, tiles="CartoDB positron")
        geojson_layer(bbox_geom, "#e74c3c", "bbox").add_to(m)

        if dataset_id not in mapping_idx.index:
            folium.LayerControl().add_to(m)
            ipd.display(Label("⚠ No zone matched — bbox only"), m)
            return

        match     = mapping_idx.loc[dataset_id]
        zone_id   = match["zone_id"]
        iou_score = match["iou_score"]
        zone_geom = zones.loc[zone_id, "geometry"] if zone_id in zones.index else None

        if zone_geom is not None:
            geojson_layer(zone_geom, "#2980b9", zone_id).add_to(m)

        folium.LayerControl().add_to(m)
        ipd.display(
            Label(f"📍 {row['title'][:80]}"),
            Label(f"zone: {zone_id}   |   iou_score: {iou_score:.3f}"),
            m,
        )

id_input = Text(
    placeholder="dataset id…",
    layout={"width": "380px"},
    continuous_update=False,  # only fires on Enter / focus-out, not every keystroke
)
btn = Button(description="Show", button_style="primary")

btn.on_click(lambda _: show_dataset(id_input.value))
id_input.observe(lambda change: show_dataset(change["new"]), names="value")

VBox([HBox([id_input, btn]), out])